In [ ]:
# ── Cell 1: Load config and read product_performance from S3 ─────────────────
exec(open("/Workspace/ecommerce_retail/config.py").read())

df_products = spark.read \
    .option("fs.s3a.access.key", ACCESS_KEY) \
    .option("fs.s3a.secret.key", SECRET_KEY) \
    .option("fs.s3a.session.token", SESSION_TOKEN) \
    .option("fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.TemporaryAWSCredentialsProvider") \
    .parquet(f"{S3_GOLD_BASE}/product_performance/")

df_products.createOrReplaceTempView("product_performance")
print(f"✅ {df_products.count()} rows loaded")
df_products.printSchema()

In [ ]:
# ── Cell 2: Install libraries ─────────────────────────────────────────────────
%pip install plotly pandas

In [ ]:
# ── Cell 3: Imports ───────────────────────────────────────────────────────────
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [ ]:
# ── Cell 4: KPI Summary Table ─────────────────────────────────────────────────
df = spark.sql("""
    SELECT
        COUNT(DISTINCT product_id)          AS total_products,
        COUNT(DISTINCT category)            AS total_categories,
        COUNT(DISTINCT brand)               AS total_brands,
        SUM(units_sold)                     AS total_units_sold,
        ROUND(SUM(revenue), 2)              AS total_revenue,
        ROUND(AVG(avg_review_rating), 2)    AS overall_avg_rating,
        ROUND(SUM(profit_estimate), 2)      AS total_profit_estimate,
        ROUND(AVG(product_price), 2)        AS avg_product_price
    FROM product_performance
""").toPandas()

kpi = df.T.reset_index()
kpi.columns = ["KPI", "Value"]

fig = go.Figure(data=[go.Table(
    header=dict(
        values=["<b>KPI</b>", "<b>Value</b>"],
        fill_color="#2E3A59",
        font=dict(color="white", size=13),
        align="left",
        height=35
    ),
    cells=dict(
        values=[kpi["KPI"], kpi["Value"]],
        fill_color=[["#F1EFE8", "#FFFFFF"] * len(kpi)],
        font=dict(size=12),
        align="left",
        height=30
    )
)])
fig.update_layout(title="🛒 Product Performance — KPI Summary")
fig.show()

In [ ]:
# ── Cell 5: Chart 1 — Revenue by Category (Bar) ───────────────────────────────
df = spark.sql("""
    SELECT
        category,
        ROUND(SUM(revenue), 2)      AS total_revenue,
        SUM(units_sold)             AS total_units_sold,
        COUNT(DISTINCT product_id)  AS product_count
    FROM product_performance
    GROUP BY category
    ORDER BY total_revenue DESC
""").toPandas()

fig = px.bar(
    df,
    x="category",
    y="total_revenue",
    title="📦 Total Revenue by Category",
    color="category",
    color_discrete_sequence=px.colors.qualitative.Set2,
    text="total_revenue",
    hover_data=["total_units_sold", "product_count"]
)
fig.update_traces(textposition="outside")
fig.update_layout(
    showlegend=False,
    xaxis_title="Category",
    yaxis_title="Total Revenue ($)"
)
fig.show()

In [ ]:
# ── Cell 6: Chart 2 — Profit Estimate by Category ─────────────────────────────
df = spark.sql("""
    SELECT
        category,
        ROUND(SUM(profit_estimate), 2)  AS total_profit,
        ROUND(SUM(revenue), 2)          AS total_revenue,
        ROUND(
            SUM(profit_estimate) / NULLIF(SUM(revenue), 0) * 100
        , 2)                            AS profit_margin_pct
    FROM product_performance
    GROUP BY category
    ORDER BY total_profit DESC
""").toPandas()

fig = px.bar(
    df,
    x="category",
    y="total_profit",
    title="💰 Total Profit Estimate by Category",
    color="profit_margin_pct",
    color_continuous_scale="Greens",
    text="total_profit",
    hover_data=["profit_margin_pct"]
)
fig.update_traces(textposition="outside")
fig.update_layout(
    xaxis_title="Category",
    yaxis_title="Profit Estimate ($)"
)
fig.show()

In [ ]:
# ── Cell 7: Chart 3 — Top 15 Products by Revenue (Horizontal Bar) ─────────────
df = spark.sql("""
    SELECT
        product_name,
        category,
        ROUND(revenue, 2)         AS revenue,
        units_sold,
        ROUND(profit_estimate, 2) AS profit_estimate
    FROM product_performance
    ORDER BY revenue DESC
    LIMIT 15
""").toPandas()

fig = px.bar(
    df,
    x="revenue",
    y="product_name",
    orientation="h",
    title="🏆 Top 15 Products by Revenue",
    color="category",
    color_discrete_sequence=px.colors.qualitative.Pastel,
    text="revenue",
    hover_data=["units_sold", "profit_estimate"]
)
fig.update_traces(textposition="outside")
fig.update_layout(
    yaxis={"categoryorder": "total ascending"},
    xaxis_title="Revenue ($)",
    yaxis_title="Product"
)
fig.show()

In [ ]:
# ── Cell 8: Chart 4 — Units Sold by Category (Donut) ─────────────────────────
df = spark.sql("""
    SELECT
        category,
        SUM(units_sold) AS total_units_sold
    FROM product_performance
    GROUP BY category
    ORDER BY total_units_sold DESC
""").toPandas()

fig = px.pie(
    df,
    names="category",
    values="total_units_sold",
    title="📊 Units Sold by Category",
    hole=0.4,
    color_discrete_sequence=px.colors.qualitative.Set3
)
fig.update_traces(textinfo="percent+label+value")
fig.show()

In [ ]:
# ── Cell 9: Chart 5 — Avg Review Rating by Category ──────────────────────────
df = spark.sql("""
    SELECT
        category,
        ROUND(AVG(avg_review_rating), 2) AS avg_rating,
        SUM(review_count)                AS total_reviews
    FROM product_performance
    WHERE avg_review_rating IS NOT NULL
    GROUP BY category
    ORDER BY avg_rating DESC
""").toPandas()

fig = px.bar(
    df,
    x="category",
    y="avg_rating",
    title="⭐ Average Review Rating by Category",
    color="avg_rating",
    color_continuous_scale="YlGn",
    text="avg_rating",
    hover_data=["total_reviews"]
)
fig.update_traces(textposition="outside")
fig.update_layout(
    xaxis_title="Category",
    yaxis_title="Avg Rating",
    yaxis=dict(range=[0, 5])
)
fig.show()

In [ ]:
# ── Cell 10: Chart 6 — Revenue vs Profit by Category (Grouped Bar) ───────────
df = spark.sql("""
    SELECT
        category,
        ROUND(SUM(revenue), 2)         AS total_revenue,
        ROUND(SUM(profit_estimate), 2) AS total_profit
    FROM product_performance
    GROUP BY category
    ORDER BY total_revenue DESC
""").toPandas()

fig = go.Figure()
fig.add_trace(go.Bar(
    name="Revenue",
    x=df["category"],
    y=df["total_revenue"],
    marker_color="#636EFA",
    text=df["total_revenue"],
    textposition="outside"
))
fig.add_trace(go.Bar(
    name="Profit Estimate",
    x=df["category"],
    y=df["total_profit"],
    marker_color="#2ECC71",
    text=df["total_profit"],
    textposition="outside"
))
fig.update_layout(
    barmode="group",
    title="📈 Revenue vs Profit Estimate by Category",
    xaxis_title="Category",
    yaxis_title="Amount ($)",
    legend_title="Metric"
)
fig.show()

In [ ]:
# ── Cell 11: Chart 7 — Price vs Rating Scatter ───────────────────────────────
df = spark.sql("""
    SELECT
        product_name,
        category,
        product_price,
        avg_review_rating,
        units_sold,
        ROUND(revenue, 2) AS revenue
    FROM product_performance
    WHERE avg_review_rating IS NOT NULL
      AND product_price IS NOT NULL
""").toPandas()

fig = px.scatter(
    df,
    x="product_price",
    y="avg_review_rating",
    color="category",
    size="units_sold",
    hover_name="product_name",
    title="🔵 Price vs Avg Review Rating (bubble size = units sold)",
    color_discrete_sequence=px.colors.qualitative.Set1,
    labels={
        "product_price":     "Product Price ($)",
        "avg_review_rating": "Avg Review Rating"
    }
)
fig.update_layout(
    yaxis=dict(range=[0, 5]),
    legend_title="Category"
)
fig.show()

In [ ]:
# ── Cell 12: Chart 8 — Top 15 Products by Profit (Horizontal Bar) ─────────────
df = spark.sql("""
    SELECT
        product_name,
        category,
        ROUND(profit_estimate, 2) AS profit_estimate,
        ROUND(revenue, 2)         AS revenue,
        units_sold
    FROM product_performance
    ORDER BY profit_estimate DESC
    LIMIT 15
""").toPandas()

fig = px.bar(
    df,
    x="profit_estimate",
    y="product_name",
    orientation="h",
    title="💎 Top 15 Products by Profit Estimate",
    color="category",
    color_discrete_sequence=px.colors.qualitative.Vivid,
    text="profit_estimate",
    hover_data=["revenue", "units_sold"]
)
fig.update_traces(textposition="outside")
fig.update_layout(
    yaxis={"categoryorder": "total ascending"},
    xaxis_title="Profit Estimate ($)",
    yaxis_title="Product"
)
fig.show()

In [ ]:
# ── Cell 13: Chart 9 — Top 10 Brands by Revenue + Units Sold ─────────────────
df = spark.sql("""
    SELECT
        brand,
        ROUND(SUM(revenue), 2) AS total_revenue,
        SUM(units_sold)        AS total_units_sold,
        COUNT(product_id)      AS product_count
    FROM product_performance
    GROUP BY brand
    ORDER BY total_revenue DESC
    LIMIT 10
""").toPandas()

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Revenue by Brand ($)", "Units Sold by Brand")
)

fig.add_trace(
    go.Bar(
        x=df["brand"], y=df["total_revenue"],
        marker_color="#636EFA",
        text=df["total_revenue"],
        textposition="outside",
        name="Revenue"
    ),
    row=1, col=1
)
fig.add_trace(
    go.Bar(
        x=df["brand"], y=df["total_units_sold"],
        marker_color="#EF553B",
        text=df["total_units_sold"],
        textposition="outside",
        name="Units Sold"
    ),
    row=1, col=2
)
fig.update_layout(
    title_text="🏷️ Top 10 Brands — Revenue vs Units Sold",
    showlegend=False,
    height=450
)
fig.show()

In [ ]:
# ── Cell 14: Chart 10 — Subcategory Treemap ──────────────────────────────────
df = spark.sql("""
    SELECT
        category,
        subcategory,
        ROUND(SUM(revenue), 2) AS total_revenue,
        SUM(units_sold)        AS total_units_sold
    FROM product_performance
    WHERE subcategory IS NOT NULL
    GROUP BY category, subcategory
""").toPandas()

fig = px.treemap(
    df,
    path=["category", "subcategory"],
    values="total_revenue",
    color="total_revenue",
    color_continuous_scale="Blues",
    title="🗺️ Revenue by Category → Subcategory (Treemap)"
)
fig.update_layout(margin=dict(t=50, l=25, r=25, b=25))
fig.show()